In [0]:
from pyspark.sql.functions import *
from delta.tables import *
from pyspark.sql.window import *

In [0]:
%run "./01_connectivity"

Connectivity configured successfully!


In [0]:
# Create gold schema and volume
spark.sql("CREATE SCHEMA IF NOT EXISTS databricks_generalhospital.gold")
print("Gold schema created!")

Gold schema created!


In [0]:
# Read patients from silver volume
volume_path = "/Volumes/databricks_generalhospital/silver/silver_volume"
df_patients_silver = spark.read.parquet(f"{volume_path}/patients/")
print(f"Patients loaded from Silver: {df_patients_silver.count()} rows")
df_patients_silver.display()

Patients loaded from Silver: 8 rows


master_patient_id,patient_address_number,patient_address_street,patient_address_full,patient_city,patient_zip_code,patient_county,patient_state,patient_country,patient_latitude,patient_longitude,patient_name,patient_gender,patient_lace_score,patient_geography_loaded,patient_dob,patient_marital_status,patient_primary_language,patient_is_citizen_flag,patient_is_deceased_flag,patient_pcp_type,patient_pcp_id,patient_ethnicity,ingestion_timestamp
105054,164,Allendale Rd,164 Allendale Rd,King of Prussia,19406,Montgomery,PA,US,40.0909754,-75.3829461,Keelby Biaggioni,Male,4,Yes,2000-01-20T00:00:00Z,Other,Unknown,N,N,P,3983,Hispanic or Latino,2026-06-22T10:57:53.166Z
103569,124,E Lancaster Ave,124 E Lancaster Ave,Wayne,19087,Delaware,PA,US,40.0436,-75.3897,Giana Ironside,Female,18,Yes,2000-01-25T00:00:00Z,Other,Unknown,N,N,P,1,White,2026-06-22T10:57:53.166Z
105550,300,Montgomery Ave,300 Montgomery Ave,Phoenixville,19460,Montgomery,PA,US,40.1309388,-75.4607432,Willard Bealing,Male,5,Yes,2000-02-16T00:00:00Z,Other,Unknown,N,N,P,3983,Hispanic or Latino,2026-06-22T10:57:53.166Z
105551,926,Hamilton Rd,926 Hamilton Rd,Collegeville,19426,Montgomery,PA,US,40.2121,-75.4594,Jehanna Cathersides,Female,8,Yes,2001-06-04T00:00:00Z,Other,ENGLISH,N,N,R,3980,Asian or Pacific Islander,2026-06-22T10:57:53.166Z
100372,484,N West St,484 N West St,Doylestown,18901,Bucks,PA,US,40.3101062,-75.1437833,Miller Elby,Male,13,Yes,2000-10-19T00:00:00Z,Other,Unknown,N,N,P,3983,Hispanic or Latino,2026-06-22T10:57:53.166Z
103569,124,E Lancaster Ave,124 E Lancaster Ave,New York,19087,Delaware,PA,US,40.0436,-75.3897,Giana Ironside,Female,18,Yes,2000-01-25T00:00:00Z,Married,Unknown,N,N,P,1,White,2026-06-22T10:57:53.166Z
101224,492,Richlandtown Pike,492 Richlandtown Pike,Quakertown,18951,Bucks,PA,US,40.4624236,-75.3239025,Claretta Glasper,Female,7,Yes,2003-04-08T00:00:00Z,Other,ENGLISH,N,N,R,4152,Black or African American,2026-06-22T10:57:53.166Z
107165,7949,Williams Ave,7949 Williams Ave,Philadelphia,19150,Philadelphia,PA,US,40.0720554,-75.1668429,Shaw Bohlsen,Male,4,Yes,2000-04-03T00:00:00Z,Other,Unknown,N,N,P,3983,Hispanic or Latino,2026-06-22T10:57:53.166Z


In [0]:
# Deduplicate — keep latest record per master_patient_id
window = Window.partitionBy("master_patient_id").orderBy(desc("ingestion_timestamp"))

df_patients_dedup = df_patients_silver \
    .withColumn("row_num", row_number().over(window)) \
    .filter("row_num = 1") \
    .drop("row_num")

print(f"Before dedup: {df_patients_silver.count()} rows")
print(f"After dedup:  {df_patients_dedup.count()} rows")

Before dedup: 8 rows
After dedup:  7 rows


In [0]:
# Step 1 — Add hash column to source (silver)
df_patients_hashed = df_patients_dedup.withColumn("hash_value",
    md5(concat_ws("||",
        col("patient_name"),
        col("patient_city"),
        col("patient_marital_status"),
        col("patient_gender"),
        col("patient_primary_language"),
        col("patient_is_deceased_flag")
    ))
).withColumn("last_updated", current_timestamp())

In [0]:
# Add hash + last_updated to patients and overwrite gold
df_patients_hashed.write.option("mergeSchema", "true") \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("databricks_generalhospital.gold.patients_gold")

print("Gold table recreated with hash column!")

Gold table recreated with hash column!


In [0]:
# Step 2 — Load Gold Delta Table
target_table = DeltaTable.forName(spark, "databricks_generalhospital.gold.patients_gold")

# Step 3 — SCD Type 1 MERGE with hash check
target_table.alias("target").merge(
    df_patients_hashed.alias("source"),
    "target.master_patient_id = source.master_patient_id"
).whenMatchedUpdate(
    condition="target.hash_value != source.hash_value",
    set={
        "patient_name"             : "source.patient_name",
        "patient_city"             : "source.patient_city",
        "patient_marital_status"   : "source.patient_marital_status",
        "patient_gender"           : "source.patient_gender",
        "patient_primary_language" : "source.patient_primary_language",
        "patient_is_deceased_flag" : "source.patient_is_deceased_flag",
        "patient_dob"              : "source.patient_dob",
        "patient_state"            : "source.patient_state",
        "patient_zip_code"         : "source.patient_zip_code",
        "hash_value"               : "source.hash_value",
        "last_updated"             : "source.last_updated"
    }
).whenNotMatchedInsertAll() \
 .execute()

print("SCD Type 1 MERGE with hash completed!")

SCD Type 1 MERGE with hash completed!


In [0]:
# Verify Gold Delta table

df_gold = spark.table("databricks_generalhospital.gold.patients_gold")
print(f"Gold table row count: {df_gold.count()}")
df_gold.display()

Gold table row count: 7


master_patient_id,patient_address_number,patient_address_street,patient_address_full,patient_city,patient_zip_code,patient_county,patient_state,patient_country,patient_latitude,patient_longitude,patient_name,patient_gender,patient_lace_score,patient_geography_loaded,patient_dob,patient_marital_status,patient_primary_language,patient_is_citizen_flag,patient_is_deceased_flag,patient_pcp_type,patient_pcp_id,patient_ethnicity,ingestion_timestamp,hash_value,last_updated
100372,484,N West St,484 N West St,Doylestown,18901,Bucks,PA,US,40.3101062,-75.1437833,Miller Elby,Male,13,Yes,2000-10-19T00:00:00Z,Other,Unknown,N,N,P,3983,Hispanic or Latino,2026-06-22T10:57:53.166Z,6f267bb63b32697a5576ce84297f756e,2026-06-22T13:46:38.338Z
101224,492,Richlandtown Pike,492 Richlandtown Pike,Quakertown,18951,Bucks,PA,US,40.4624236,-75.3239025,Claretta Glasper,Female,7,Yes,2003-04-08T00:00:00Z,Other,ENGLISH,N,N,R,4152,Black or African American,2026-06-22T10:57:53.166Z,c34fc7838f887aefb41095837be3fc03,2026-06-22T13:46:38.338Z
103569,124,E Lancaster Ave,124 E Lancaster Ave,Wayne,19087,Delaware,PA,US,40.0436,-75.3897,Giana Ironside,Female,18,Yes,2000-01-25T00:00:00Z,Other,Unknown,N,N,P,1,White,2026-06-22T10:57:53.166Z,41a5edb47d9e32d82be3b0eb9a6fbef5,2026-06-22T13:46:38.338Z
105054,164,Allendale Rd,164 Allendale Rd,King of Prussia,19406,Montgomery,PA,US,40.0909754,-75.3829461,Keelby Biaggioni,Male,4,Yes,2000-01-20T00:00:00Z,Other,Unknown,N,N,P,3983,Hispanic or Latino,2026-06-22T10:57:53.166Z,aa8ba453dcf6d3b084c77c40c5cc9cd6,2026-06-22T13:46:38.338Z
105550,300,Montgomery Ave,300 Montgomery Ave,Phoenixville,19460,Montgomery,PA,US,40.1309388,-75.4607432,Willard Bealing,Male,5,Yes,2000-02-16T00:00:00Z,Other,Unknown,N,N,P,3983,Hispanic or Latino,2026-06-22T10:57:53.166Z,578237a932cc1d461a1523d589d16f2e,2026-06-22T13:46:38.338Z
105551,926,Hamilton Rd,926 Hamilton Rd,Collegeville,19426,Montgomery,PA,US,40.2121,-75.4594,Jehanna Cathersides,Female,8,Yes,2001-06-04T00:00:00Z,Other,ENGLISH,N,N,R,3980,Asian or Pacific Islander,2026-06-22T10:57:53.166Z,ceca4b4cc0ce1227a0e8ecf79f5ec6a1,2026-06-22T13:46:38.338Z
107165,7949,Williams Ave,7949 Williams Ave,Philadelphia,19150,Philadelphia,PA,US,40.0720554,-75.1668429,Shaw Bohlsen,Male,4,Yes,2000-04-03T00:00:00Z,Other,Unknown,N,N,P,3983,Hispanic or Latino,2026-06-22T10:57:53.166Z,5a28cdf8243431ed397e326cc25107c1,2026-06-22T13:46:38.338Z
